# Uninformed Search Techniques
---
Now, we are going to explore a uninformed search technique named Breadth-First Search (BFS), applied on a graph. The implementation is based on the pseudocode provided in the lecture slides from Canva.

In [ ]:
import os

from pathlib import Path
from typing import List, Dict
from abc import ABC, abstractmethod

In [2]:
# Mount my Google Drive to have acces to my files. This code snippet is only 
# for Google Colab, it won't work on your local machine.
discriminant = os.environ.get('COLAB_JUPYTER_IP', None)
dataPath = '/My Drive/UTS/data'
mount = 'G:'

if discriminant is not None:
    from google.colab import drive
    drive.mount('/content/drive')
    del mount
    mount = '/content/drive'

dataPath = Path(mount + dataPath).resolve()

In [ ]:
class Node:
    def __init__(self, node_id: str, data: Dict = {}):
        # The constructor function takes up to two values: a, unique identifier for each node, 
        # that should be a labeling integer, and a dictionary of properties that could be 
        # either empty  
        self.id = node_id
        self.data = data

    def __hash__(self) -> int:
        # Implementing __hash__ allows Node objects to be used in hash-based collections 
        # like sets and dictionaries, ensuring uniqueness is determined by the node_id.
        return hash(self.id)

    def __eq__(self, other: object) -> bool:
        # This methods check if the instance of a node is equal to another instance of a Node
        # object by checkir their ids
        if not isinstance(other, Node):
            return False
        return self.id == other.id

    def __repr__(self) -> str:
        # Shows a string representation of the node to be called using the builtin method 
        # repr(Node)
        return f"Node({self.id})"


In [ ]:
# We implement the BaseGraph interface to ensure the future Graph objects follow the rules we 
# define for them 
class BaseGraph(ABC):
    @abstractmethod
    def add_node(self, node: Node) -> None:
        pass

    @abstractmethod
    def add_edge(self, node_1: Node, node_2: Node, weight: int = 1) -> None:
        pass

    @abstractmethod
    def get_neighbors(sef, node: Node) -> List[Node]:
        pass

In [ ]:
class MatrixGraph(BaseGraph):
    def __init__(self, directed: bool = False, verbose: bool = True):
        # The constructor function takes one value: a boolean value for verbosity. The other values
        # are initialized to empty lists and dictionaries and are modified as the graph is built.
        self.directed = directed
        self.verbose = verbose
        self.matrix: List[List[int]] = []
        self.nodes: List[Node] = []
        self.nodes_index: Dict[str, int] = {}

    def messenger(self, message: str) -> None:
        # If verbose is True, print the message
        if self.verbose:
            print(message)
        pass

    def add_node(self, node: Node) -> None:
        # Check if the node already exists within the graph
        if node.id in self.nodes_index:
            self.messenger(f'[WARN] - Node({node.id}) already exists!')
        
        # Get the index of the new node
        new_node_index = len(self.nodes)

        # Add the node to the graph
        self.nodes.append(node)
        self.nodes_index.setdefault(node.id, new_node_index)
        
        # To get the new matrix we need to grow the matrix by one column and one row. 
        # Initialize a new column of 0s and append it to all existing rows. Then, we add a new row of 0s to the matrix. 
        for row in self.matrix:
            row.append(0)
            
        new_row = [0] * len(self.nodes)
        self.matrix.append(new_row)
    
    def add_edge(self, node_1: Node, node_2: Node, weight: int = 1):
        # Get the indices of the nodes. First, we check if they exist in the graph.
        idx, jdx = None, None

        try:
            if idx is None or jdx is None:
                idx = self.nodes_index[node_1.id]
                jdx = self.nodes_index[node_2.id]
        except KeyError as err:
            raise ValueError(f"Both nodes must be added to the graph first! Missing: {err}")
        
        # Set the weight in the matrix
        self.matrix[idx][jdx] = weight
        
        # Consider if the graph is directed or not to decide whether  
        if not self.directed:
            self.matrix[jdx][idx] = weight

    def get_neighbors(self, node: Node) -> list[Node]:
        # This function returns the neighbors of a given node within the graph. 
        # First, we check if the fiven node is part/connected to the graph
        if node.id not in self.nodes_index:
            return []
        
        # Then, we determine if the index valule of the node according with its id
        node_idx = self.nodes_index[node.id]
        
        neighbors = []
        row = self.matrix[node_idx]
        
        for col_idx, weight in enumerate(row):
            if weight != 0:
                neighbors.append(self.nodes[col_idx])
                
        return neighbors

    def print_matrix(self):
        print("    " + " ".join([n.id for n in self.nodes]))
        for i, row in enumerate(self.matrix):
            print(f"{self.nodes[i].id} | {row}")


In [ ]:
class ListGraph(BaseGraph):
    def __init__(self):
        # The core data structure: A dictionary mapping a Node -> List of its Neighbors (and weights)
        # We store tuples (Node, weight) in the list so we know the cost of the edge.
        self.adjacency_list: dict[Node, list[tuple[Node, int]]] = {}

    def add_node(self, node: Node):
        # We only add the node if it doesn't already exist as a key.
        # This is where __hash__ and __eq__ are doing the hard work behind the scenes!
        if node not in self.adjacency_list:
            # Initialize with an empty list of neighbors
            self.adjacency_list[node] = []

    def add_edge(self, node1: Node, node2: Node, weight: int = 1, directed: bool = False):
        # Good Practice: Guard clauses. Ensure both nodes exist in the graph first.
        if node1 not in self.adjacency_list or node2 not in self.adjacency_list:
            raise ValueError("Both nodes must be added to the graph first!")

        # 1. Add node2 to node1's list of neighbors
        self.adjacency_list[node1].append((node2, weight))

        # 2. If it is an undirected graph (two-way street), add node1 to node2's list of neighbors!
        if not directed:
            self.adjacency_list[node2].append((node1, weight))

    def get_neighbors(self, node: Node) -> list[Node]:
        # Simple lookup!
        # Safety check: Return an empty list if the node isn't in our graph.
        if node not in self.adjacency_list:
            return []
            
        # Our dictionary stores tuples of (neighbor_node, weight). 
        # For this function, we just want to return a list of the neighbor Nodes themselves.
        # We can extract the Nodes using a simple list comprehension.
        return [neighbor for neighbor, weight in self.adjacency_list[node]]
        
    def get_neighbors_with_weights(self, node: Node) -> list[tuple[Node, int]]:
        # Sometimes (like in A* or Dijkstra's algorithm) we need to know the edge cost too!
        if node not in self.adjacency_list:
            return []
        return self.adjacency_list[node]

    # --- Bonus visualization method ---
    def print_list(self):
        for node, neighbors in self.adjacency_list.items():
            # Format nicely: Node(A) -> Node(B)(cost:1), Node(C)(cost:5)
            neighbor_strings = [f"{n.id}(cost:{w})" for n, w in neighbors]
            print(f"{node.id} -> {', '.join(neighbor_strings)}")
